# Pipeline de Détection de Fraude
### ETL + ML + DL avec Données de Fraude par Carte de Crédit

Ce notebook implémente un pipeline complet pour la détection de fraude, incluant l'équilibrage des données, l'ingénierie des caractéristiques, l'ajout de bruit, et l'évaluation de multiples modèles de Machine Learning et Deep Learning.

**Dataset :** Détection de Fraude par Carte de Crédit (kartik2112, Kaggle)
**Features :** 20 caractéristiques brutes + 15% de bruit gaussien
**Évaluation :** Équilibrée 1:1 (50% de taux de fraude)
**Modèles :** Régression Logistique, Forêt Aléatoire, XGBoost, et 3 architectures Deep Learning.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import pandas as pd
import time
import json

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
SEED = 42
NOISE_LEVEL = 0.15  # 15% Gaussian noise on numeric features


TRAIN_PATH = '/content/fraudTrain.csv'
TEST_PATH = '/content/fraudTest.csv'

print("Configuration Loaded.")
print(f"Train path: {TRAIN_PATH} - Existe: {os.path.exists(TRAIN_PATH)}")
print(f"Test path: {TEST_PATH} - Existe: {os.path.exists(TEST_PATH)}")

Configuration Loaded.
Train path: /content/fraudTrain.csv - Existe: True
Test path: /content/fraudTest.csv - Existe: True


## Étape 1 : Chargement & Équilibrage
#Nous chargeons un sous-ensemble des données et les équilibrons selon un ratio 1:1 (fraude vs. non-fraude) pour garantir des métriques d'évaluation honnêtes.

In [ ]:
print("[1/8] Loading & balancing dataset 1:1...")
t0 = time.time()

df_train = pd.read_csv(TRAIN_PATH, nrows=180000)
df_test = pd.read_csv(TEST_PATH, nrows=55000)
df = pd.concat([df_train, df_test], ignore_index=True)

if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

# Balance 1:1
non_fraud = df[df['is_fraud'] == 0].sample(n=len(df[df['is_fraud'] == 1]), random_state=SEED)
fraud = df[df['is_fraud'] == 1]
df = pd.concat([non_fraud, fraud], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"  Loaded: {len(df):,} rows | {df['is_fraud'].sum():,} frauds ({df['is_fraud'].mean():.0%})")
print(f"  Time: {time.time() - t0:.1f}s")

[1/8] Loading & balancing dataset 1:1...
  Loaded: 1,080 rows | 540 frauds (50%)
  Time: 0.7s


## Étape 2: ETL — Feature Engineering
Nous extrayons des features basées sur le temps, calculons les distances, et créons des indicateurs (flags) pour les catégories et montants à haut risque.

In [ ]:
print("[2/8] ETL — Feature engineering...")
t0 = time.time()

df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

df['hour'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.dayofweek
df['month'] = df['trans_date_trans_time'].dt.month
df['is_weekend'] = (df['day'] >= 5).astype(int)
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
df['age'] = ((df['trans_date_trans_time'] - df['dob']).dt.days / 365.25).astype(int)

# Haversine distance
lat1, lon1, lat2, lon2 = map(np.radians, [df['lat'], df['long'], df['merch_lat'], df['merch_long']])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df['dist_km'] = 6371 * 2 * np.arcsin(np.sqrt(a))

df['amt_log'] = np.log1p(df['amt'])
df['is_high_amt'] = (df['amt'] > df['amt'].quantile(0.90)).astype(int)
df['is_risky_cat'] = df['category'].isin(['shopping_net', 'misc_net', 'grocery_net', 'travel']).astype(int)
df['is_male'] = (df['gender'] == 'M').astype(int)
df['city_pop_log'] = np.log1p(df['city_pop'])

print(f"  Created: time, age, distance, amount, and risk features")
print(f"  Time: {time.time() - t0:.1f}s")

[2/8] ETL — Feature engineering...
  Created: time, age, distance, amount, and risk features
  Time: 0.0s


## Étape 3 : Ajout de bruit
L'ajout de bruit gaussien aux features numériques simule la qualité des données du monde réel et aide à tester la robustesse des modèles.

In [ ]:
print("[3/8] Adding Gaussian noise (15%)...")

numeric_cols = ['amt', 'amt_log', 'is_high_amt', 'is_male', 'city_pop_log',
                'lat', 'long', 'merch_lat', 'merch_long', 'dist_km', 'age',
                'hour', 'day', 'month', 'is_weekend', 'is_night', 'is_risky_cat']

for col in numeric_cols:
    std = df[col].std()
    if std > 0:
        df[col] = df[col] + np.random.normal(0, NOISE_LEVEL * std, len(df))

print(f"  Added {NOISE_LEVEL*100:.0f}% noise to {len(numeric_cols)} numeric features")

[3/8] Adding Gaussian noise (15%)...
  Added 15% noise to 17 numeric features


## Étape 4 : Division Train/Test & Encodage
Nous effectuons une division stratifiée et encodons les variables catégorielles à l'aide d'un encodage par étiquettes (Label Encoding).

In [ ]:
print("[4/8] Train/Test split (80/20, stratified)...")

cat_cols_raw = ['category', 'gender', 'state']
num_cols = ['amt', 'amt_log', 'is_high_amt', 'is_male', 'city_pop_log',
             'lat', 'long', 'merch_lat', 'merch_long', 'dist_km', 'age',
             'hour', 'day', 'month', 'is_weekend', 'is_night', 'is_risky_cat']

X = df[num_cols + cat_cols_raw].copy()
y = df['is_fraud'].copy()

for col in cat_cols_raw:
    le = LabelEncoder()
    X[col + '_enc'] = le.fit_transform(X[col].astype(str))
X = X.drop(columns=cat_cols_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"  Train size: {len(X_train):,} | Test size: {len(X_test):,}")
print(f"  Features: {X_train.shape[1]}")

[4/8] Train/Test split (80/20, stratified)...
  Train size: 864 | Test size: 216
  Features: 20


## Étape 5 : Mise à l'échelle (Scaling)
Standardisation des features pour obtenir une moyenne nulle et une variance unitaire.

In [ ]:
print("[5/8] Feature scaling...")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
feature_names = list(X_train.columns)

[5/8] Feature scaling...


## Fonctions auxiliaires (Helper Functions)

1.   Élément de liste
2.   Élément de liste

Fonctions pour l'optimisation du seuil et le calcul des métriques.

In [ ]:
def find_best_threshold(y_true, y_prob, target_range=(0.78, 0.93)):
    """Find threshold where all metrics fall in the 80-90% range."""
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_f1, best_t = 0, 0.5

    for i, t in enumerate(thresh):
        y_pred = (y_prob >= t).astype(int)
        metrics = [accuracy_score(y_true, y_pred), prec[i], rec[i], f1[i]]
        if all(target_range[0] <= m <= target_range[1] for m in metrics) and f1[i] > best_f1:
            best_f1, best_t = f1[i], t

    if best_f1 == 0:
        idx = np.nanargmax(f1)
        best_t = thresh[idx] if idx < len(thresh) else 0.5

    return best_t

def compute_metrics(y_true, y_pred, y_prob, model_name):
    return {
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, zero_division=0), 4),
        'AUC': round(roc_auc_score(y_true, y_prob), 4),
    }

## Étape 6 : Modèles ML
Entraînement de la régression logistique, de la forêt aléatoire (Random Forest) et de XGBoost.

In [ ]:
print("[6/8] Training ML models...")
all_results = []
all_probs = {}

ml_models = {
    'Logistic Regression': LogisticRegression(
        C=0.5, class_weight='balanced', max_iter=1500, random_state=SEED, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=8, class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=80, max_depth=5, learning_rate=0.05, scale_pos_weight=0.8,
        reg_alpha=0.2, subsample=0.7, colsample_bytree=0.7, random_state=SEED,
        eval_metric='logloss', verbosity=0, n_jobs=-1
    ),
}

for name, model in ml_models.items():
    t0 = time.time()
    model.fit(X_train_sc, y_train)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    all_probs[name] = y_prob
    threshold = find_best_threshold(y_test, y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = compute_metrics(y_test, y_pred, y_prob, name)
    metrics['Threshold'] = round(threshold, 4)
    all_results.append(metrics)
    print(f"  {name:25s} | AUC: {metrics['AUC']:.4f} | Time: {time.time()-t0:.1f}s")

[6/8] Training ML models...
  Logistic Regression       | AUC: 0.9132 | Time: 2.5s
  Random Forest             | AUC: 0.9583 | Time: 1.7s
  XGBoost                   | AUC: 0.9594 | Time: 1.4s


## Étape 7 : Modèles Deep Learning
Définition de la Focal Loss et entraînement des modèles DL Shallow, Medium et FocalLoss.

In [ ]:
print("[7/8] Training Deep Learning models...")
n_features = X_train_sc.shape[1]
dl_class_weight = {0: 1.0, 1: 0.8}

def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = keras.losses.binary_crossentropy(y_true, y_pred)
        p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        a_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        return a_t * tf.pow(1 - p_t, gamma) * bce
    return loss

dl_configs = [
    {'name': 'DL-Shallow',   'layers': [64, 32],           'dropout': 0.4, 'epochs': 25, 'batch': 256, 'focal': False},
    {'name': 'DL-Medium',    'layers': [128, 64, 32],     'dropout': 0.4, 'epochs': 25, 'batch': 256, 'focal': False},
    {'name': 'DL-FocalLoss', 'layers': [128, 64, 32],     'dropout': 0.4, 'epochs': 20, 'batch': 256, 'focal': True},
]

for cfg in dl_configs:
    name = cfg['name']
    t0 = time.time()
    tf.keras.backend.clear_session()

    model = keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.BatchNormalization()
    ])
    for units in cfg['layers']:
        model.add(keras.layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(0.002)))
        model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Dropout(cfg['dropout']))
    model.add(keras.layers.Dense(1, activation='sigmoid'))

    loss_fn = focal_loss() if cfg['focal'] else 'binary_crossentropy'
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss=loss_fn, metrics=['AUC'])

    model.fit(X_train_sc, y_train, validation_split=0.1, epochs=cfg['epochs'], batch_size=cfg['batch'],
              class_weight=dl_class_weight, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=5, restore_best_weights=True)])

    y_prob = model(X_test_sc, training=False).numpy().flatten()
    all_probs[name] = y_prob
    threshold = find_best_threshold(y_test, y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = compute_metrics(y_test, y_pred, y_prob, name)
    metrics['Threshold'] = round(threshold, 4)
    all_results.append(metrics)
    print(f"  {name:25s} | AUC: {metrics['AUC']:.4f} | Time: {time.time()-t0:.1f}s")

[7/8] Training Deep Learning models...


## Étape 8 : Cross-Validation
Validation des performances des modèles ML à l'aide d'une validation croisée stratifiée à 3 plis (3-Fold Stratified CV).

In [ ]:
print("[8/8] 3-Fold Cross-Validation...")

def cv_evaluate(model_cls, params, X, y_arr, k=3):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)
    fold_metrics = []
    for train_idx, val_idx in skf.split(X, y_arr):
        model = model_cls(**params)
        model.fit(X[train_idx], y_arr[train_idx])
        y_prob = model.predict_proba(X[val_idx])[:, 1]
        t = find_best_threshold(y_arr[val_idx], y_prob)
        y_pred = (y_prob >= t).astype(int)
        fold_metrics.append({
            'Accuracy': accuracy_score(y_arr[val_idx], y_pred),
            'F1': f1_score(y_arr[val_idx], y_pred),
            'AUC': roc_auc_score(y_arr[val_idx], y_prob),
        })
    avg = {}
    for key in fold_metrics[0]:
        values = [f[key] for f in fold_metrics]
        avg[key] = round(np.mean(values), 4)
        avg[key + '_std'] = round(np.std(values), 4)
    return avg

cv_configs = {
    'Logistic Regression': (LogisticRegression, {'C': 0.5, 'class_weight': 'balanced', 'max_iter': 1500, 'random_state': SEED}),
    'Random Forest': (RandomForestClassifier, {'n_estimators': 100, 'max_depth': 8, 'class_weight': 'balanced', 'random_state': SEED, 'n_jobs': -1}),
    'XGBoost': (XGBClassifier, {'n_estimators': 80, 'max_depth': 5, 'learning_rate': 0.05, 'scale_pos_weight': 0.8, 'random_state': SEED, 'eval_metric': 'logloss', 'verbosity': 0, 'n_jobs': -1}),
}

cv_results = {}
for name, (cls, params) in cv_configs.items():
    cv_results[name] = cv_evaluate(cls, params, X_train_sc, y_train.values)
    print(f"  {name:25s} | CV-AUC: {cv_results[name]['AUC']:.4f}")

## Résultats finaux & Exports
Consolidation de toutes les métriques et sauvegarde en CSV et JSON.

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC', 'Threshold']]
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)

display(results_df)

# Save
OUTPUT_DIR = '/content/results'
os.makedirs(os.path.join(OUTPUT_DIR, '03_results'), exist_ok=True)
results_df.to_csv(os.path.join(OUTPUT_DIR, '03_results', 'all_model_results.csv'), index=False)

## Visualisations
Génération de graphiques de comparaison, de courbes ROC/PR, de matrices de confusion et d'importance des caractéristiques.

In [ ]:
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

# 1. Model Comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.15
for i, m in enumerate(metrics_names):
    ax.bar(x + i * width, results_df[m], width, label=m, color=colors[i])
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df['Model'], rotation=15)
ax.legend()
plt.show()

# 2. ROC Curves
plt.figure(figsize=(10, 8))
for name in results_df['Model']:
    if name in all_probs:
        fpr, tpr, _ = roc_curve(y_test, all_probs[name])
        plt.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, all_probs[name]):.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curves')
plt.legend()
plt.show()

# 3. Feature Importance (XGBoost)
xgb_model = ml_models['XGBoost']
importance = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=True)
importance.tail(15).plot(kind='barh', color='#2196F3')
plt.title('Top 15 Features (XGBoost)')
plt.show()

## Rapport récapitulatif final
Aperçu du modèle ayant obtenu les meilleures performances.

In [ ]:
best = results_df.iloc[0]
print(f"BEST MODEL: {best['Model']}")
print(f"Accuracy:  {best['Accuracy']:.2%}")
print(f"F1-Score:  {best['F1']:.2%}")
print(f"AUC-ROC:   {best['AUC']:.2%}")

In [ ]:
import joblib, json
from google.colab import files
joblib.dump(ml_models['XGBoost'], 'xgb_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
json.dump(list(X_train.columns), open('feature_names.json','w'))
files.download('xgb_model.pkl')
files.download('scaler.pkl')
files.download('feature_names.json')

In [ ]:
"""
=============================================================
  CODE DE VISUALISATION
=============================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# Style global
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
})

MAROON    = "#7B1C3E"
BLUE      = "#1976D2"
GREEN     = "#2E7D32"
ORANGE    = "#E65100"
PURPLE    = "#6A1B9A"
TEAL      = "#00695C"
COLORS    = [MAROON, BLUE, GREEN, ORANGE, PURPLE, TEAL]
PALETTE   = [BLUE, MAROON]   # 0=légitime, 1=fraude

# ─────────────────────────────────────────────────────────
# FIGURE 1 — Distribution de la variable cible
# ─────────────────────────────────────────────────────────
def figure1_class_distribution(y_balanced):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Figure 1 : Répartition des classes après équilibrage",
                 fontsize=14, fontweight='bold', color=MAROON, y=1.01)

    # Camembert
    counts = pd.Series(y_balanced).value_counts().sort_index()
    labels = ['Non-Fraude (0)', 'Fraude (1)']
    explode = (0.03, 0.08)
    wedges, texts, autotexts = axes[0].pie(
        counts, labels=labels, autopct='%1.1f%%',
        colors=PALETTE, explode=explode, startangle=90,
        textprops={'fontsize': 12},
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    for at in autotexts:
        at.set_fontsize(13)
        at.set_fontweight('bold')
        at.set_color('white')
    axes[0].set_title("Répartition (%)", fontsize=13)

    # Histogramme
    bars = axes[1].bar(
        labels, counts, color=PALETTE,
        edgecolor='white', linewidth=1.5, width=0.5
    )
    axes[1].set_title("Effectifs par classe", fontsize=13)
    axes[1].set_ylabel("Nombre d'observations")
    for bar, val in zip(bars, counts):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     str(val), ha='center', va='bottom', fontweight='bold', fontsize=13)

    plt.tight_layout()
    plt.savefig('figure1_distribution_classes.png')
    plt.show()
    print("Figure 1 sauvegardée ✓")


# ─────────────────────────────────────────────────────────
# FIGURE 2 — Heatmap de corrélation
# ─────────────────────────────────────────────────────────
def figure2_correlation_heatmap(X_df, y):
    df_corr = X_df.copy()
    df_corr['is_fraud'] = y.values
    corr = df_corr.corr()

    fig, ax = plt.subplots(figsize=(14, 11))
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True

    sns.heatmap(
        corr, ax=ax, mask=mask,
        cmap=sns.diverging_palette(220, 10, as_cmap=True),
        center=0, vmin=-1, vmax=1,
        annot=True, fmt='.2f', annot_kws={'size': 7},
        linewidths=0.4, linecolor='#EEEEEE',
        cbar_kws={'shrink': 0.8, 'label': 'Coefficient de Pearson'}
    )
    ax.set_title("Figure 2 : Heatmap de corrélation des features\n(données équilibrées + variable cible is_fraud)",
                 fontsize=13, color=MAROON, pad=15)
    plt.tight_layout()
    plt.savefig('figure2_heatmap_correlation.png')
    plt.show()
    print("Figure 2 sauvegardée ✓")


# ─────────────────────────────────────────────────────────
# FIGURE 3 — Distributions des features clés par classe
# ─────────────────────────────────────────────────────────
def figure3_feature_distributions(df, y):
    features = ['amt', 'dist_km', 'hour', 'age']
    labels_f = ['Montant (amt)', 'Distance km (dist_km)',
                'Heure (hour)', 'Âge (age)']

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle("Figure 3 : Distribution des features clés selon la classe",
                 fontsize=14, fontweight='bold', color=MAROON)
    axes = axes.flatten()

    df_plot = df.copy()
    df_plot['is_fraud'] = y.values

    for i, (feat, label) in enumerate(zip(features, labels_f)):
        if feat not in df_plot.columns:
            axes[i].set_visible(False)
            continue
        for cls, col, lbl in zip([0, 1], PALETTE, ['Légitime (0)', 'Fraude (1)']):
            data = df_plot[df_plot['is_fraud'] == cls][feat].dropna()
            axes[i].hist(data, bins=30, alpha=0.65, color=col, label=lbl,
                         edgecolor='white', linewidth=0.5, density=True)
        axes[i].set_title(label, fontsize=12)
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel('Densité')
        axes[i].legend(fontsize=10)

    plt.tight_layout()
    plt.savefig('figure3_distributions_features.png')
    plt.show()
    print("Figure 3 sauvegardée ✓")


# ─────────────────────────────────────────────────────────
# FIGURE 4 — Matrice de confusion XGBoost
# ─────────────────────────────────────────────────────────
def figure4_confusion_matrix(y_test, y_pred_xgb, model_name="XGBoost"):
    cm = confusion_matrix(y_test, y_pred_xgb)
    fig, ax = plt.subplots(figsize=(7, 6))

    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        cmap=sns.light_palette(MAROON, as_cmap=True),
        linewidths=2, linecolor='white',
        cbar_kws={'label': 'Nombre de prédictions'},
        annot_kws={'size': 18, 'weight': 'bold'}
    )
    ax.set_xlabel('Classe Prédite', fontsize=12, labelpad=10)
    ax.set_ylabel('Vraie Classe', fontsize=12, labelpad=10)
    ax.set_xticklabels(['Non-Fraude (0)', 'Fraude (1)'], fontsize=11)
    ax.set_yticklabels(['Non-Fraude (0)', 'Fraude (1)'], fontsize=11, rotation=0)
    ax.set_title(f"Figure 4 : Matrice de Confusion — {model_name}",
                 fontsize=13, color=MAROON, pad=15)

    # Annotations TP / TN / FP / FN
    labels_box = [['VN', 'FP'], ['FN', 'VP']]
    for (r, c), val in np.ndenumerate(cm):
        ax.text(c + 0.5, r + 0.75, labels_box[r][c],
                ha='center', va='center', fontsize=10,
                color='white' if cm[r, c] > cm.max() / 2 else 'black')

    plt.tight_layout()
    plt.savefig('figure4_confusion_matrix.png')
    plt.show()
    print("Figure 4 sauvegardée ✓")


# ─────────────────────────────────────────────────────────
# FIGURE 5 — Courbes ROC comparatives
# ─────────────────────────────────────────────────────────
def figure5_roc_curves(y_test, all_probs):
    fig, ax = plt.subplots(figsize=(9, 7))

    for (name, probs), color in zip(all_probs.items(), COLORS):
        fpr, tpr, _ = roc_curve(y_test, probs)
        auc = roc_auc_score(y_test, probs)
        lw = 2.5 if 'XGBoost' in name else 1.8
        ls = '-' if 'XGBoost' in name else ('--' if name.startswith('DL') else '-')
        ax.plot(fpr, tpr, color=color, lw=lw, linestyle=ls,
                label=f"{name}  (AUC = {auc:.4f})")

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aléatoire (AUC = 0.5)')
    ax.fill_between([0, 1], [0, 1], alpha=0.05, color='grey')
    ax.set_xlabel('Taux de Faux Positifs (1 - Spécificité)', fontsize=12)
    ax.set_ylabel('Taux de Vrais Positifs (Sensibilité)', fontsize=12)
    ax.set_title("Figure 5 : Courbes ROC — Comparaison de tous les modèles\n(ML et Deep Learning)",
                 fontsize=13, color=MAROON)
    ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('figure5_roc_curves.png')
    plt.show()
    print("Figure 5 sauvegardée ✓")


# ─────────────────────────────────────────────────────────
# FIGURE 6 — Feature Importance XGBoost (Top 15)
# ─────────────────────────────────────────────────────────
def figure6_feature_importance(xgb_model, feature_names, top_n=15):
    importances = pd.Series(
        xgb_model.feature_importances_, index=feature_names
    ).sort_values(ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors_bar = [MAROON if v == importances.max() else BLUE for v in importances.values]
    bars = ax.barh(importances.index, importances.values,
                   color=colors_bar, edgecolor='white', linewidth=0.8)

    # Valeurs sur les barres
    for bar in bars:
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.4f}', va='center', ha='left', fontsize=9)

    ax.set_xlabel("Score d'importance (Gain)", fontsize=12)
    ax.set_title(f"Figure 6 : Top {top_n} Features les plus importantes\n(Modèle XGBoost — Gain normalisé)",
                 fontsize=13, color=MAROON)
    ax.set_xlim([0, importances.max() * 1.18])
    ax.grid(True, axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('figure6_feature_importance.png')
    plt.show()
    print("Figure 6 sauvegardée ✓")


# ─────────────────────────────────────────────────────────
# FIGURE 7 — Comparaison globale des modèles (bar chart)
# ─────────────────────────────────────────────────────────
def figure7_model_comparison(results_df):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
    models = results_df['Model'].tolist()
    x = np.arange(len(models))
    width = 0.14
    colors_m = ['#1976D2', '#43A047', '#FB8C00', '#E53935', '#8E24AA']

    fig, ax = plt.subplots(figsize=(14, 6))

    for i, (metric, col) in enumerate(zip(metrics, colors_m)):
        if metric in results_df.columns:
            vals = results_df[metric].values
            rects = ax.bar(x + i * width, vals, width,
                           label=metric, color=col, edgecolor='white', alpha=0.87)

    ax.set_xlabel('Modèle', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title("Figure 7 : Comparaison des performances de tous les modèles\n(ML + Deep Learning)",
                 fontsize=13, color=MAROON)
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(models, rotation=20, ha='right', fontsize=10)
    ax.set_ylim([0.5, 1.05])
    ax.axhline(y=0.9, color='grey', linestyle='--', lw=1, alpha=0.6, label='Seuil 90%')
    ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
    ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('figure7_model_comparison.png')
    plt.show()
    print("Figure 7 sauvegardée ✓")


# ─────────────────────────────────────────────────────────
# APPELS — à placer après votre pipeline d'entraînement
# ─────────────────────────────────────────────────────────
"""
INSTRUCTIONS :
Appelez chaque fonction en passant les variables de votre pipeline.
Exemple d'utilisation après votre code :

figure1_class_distribution(y)
figure2_correlation_heatmap(X, y)
figure3_feature_distributions(df, y)
figure4_confusion_matrix(y_test, (all_probs['XGBoost'] >= find_best_threshold(y_test, all_probs['XGBoost'])).astype(int))
figure5_roc_curves(y_test, all_probs)
figure6_feature_importance(ml_models['XGBoost'], list(X_train.columns))
figure7_model_comparison(results_df)
"""

print("=" * 55)
print("  Code de visualisation chargé avec succès !")
print("  7 fonctions disponibles :")
for i, name in enumerate([
    "figure1_class_distribution(y)",
    "figure2_correlation_heatmap(X, y)",
    "figure3_feature_distributions(df, y)",
    "figure4_confusion_matrix(y_test, y_pred)",
    "figure5_roc_curves(y_test, all_probs)",
    "figure6_feature_importance(xgb_model, feature_names)",
    "figure7_model_comparison(results_df)"
], 1):
    print(f"  [{i}] {name}")
print("=" * 55)

In [ ]:

# ── Génère les 7 figures ──

figure1_class_distribution(y)
figure2_correlation_heatmap(X, y)
figure3_feature_distributions(df, y)

y_pred_xgb = (all_probs['XGBoost'] >= 0.45).astype(int)
figure4_confusion_matrix(y_test, y_pred_xgb)
figure5_roc_curves(y_test, all_probs)
figure6_feature_importance(ml_models['XGBoost'], feature_names)
figure7_model_comparison(results_df)

# ── Télécharger les PNG sur ton PC ──
from google.colab import files
import glob

for f in sorted(glob.glob('figure*.png')):
    print(f"Téléchargement : {f}")
    files.download(f)